# Experiment: Mechanism Gap Matrix

Objective: compare broad positive PubMed queries against narrow integrated Nakafa Lab queries.

Success criteria:

- component-level biology should show retrieved literature;
- the fully integrated affordable `HPFH-like` plus alpha-globin-cleanup question should be visible as a gap boundary;
- the output should produce a small table that can move into a finding.

This notebook is research prioritization only. It is not medical advice.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

ROOT = Path.cwd()
DATA_DIR = "data/literature/pubmed"
SNAPSHOTS = [
    {
        "lane": "broad_hbf_induction",
        "role": "positive_component",
        "path": f"{DATA_DIR}/2026-04-27-gap-matrix-positive-hbf-induction.json",
    },
    {
        "lane": "hpfh_hbg_promoter",
        "role": "positive_component",
        "path": (f"{DATA_DIR}/2026-04-27-gap-matrix-positive-hpfh-hbg-promoter.json"),
    },
    {
        "lane": "alpha_globin_autophagy",
        "role": "positive_component",
        "path": f"{DATA_DIR}/2026-04-27-gap-matrix-positive-alpha-autophagy.json",
    },
    {
        "lane": "small_molecule_hbf",
        "role": "positive_component",
        "path": f"{DATA_DIR}/2026-04-27-gap-matrix-broad-small-molecule-hbf.json",
    },
    {
        "lane": "integrated_assay_gates",
        "role": "narrow_bridge",
        "path": f"{DATA_DIR}/2026-04-27-gap-matrix-integrated-assay-gates.json",
    },
    {
        "lane": "hpfh_like_small_molecule",
        "role": "gap_boundary",
        "path": (
            f"{DATA_DIR}/2026-04-27-novelty-boundary-hpfh-like-small-molecule.json"
        ),
    },
    {
        "lane": "hbf_alpha_autophagy_small_molecule",
        "role": "gap_boundary",
        "path": (f"{DATA_DIR}/2026-04-27-novelty-boundary-hbf-alpha-autophagy.json"),
    },
    {
        "lane": "affordable_cure_small_molecule",
        "role": "gap_boundary",
        "path": (
            f"{DATA_DIR}/"
            "2026-04-27-novelty-boundary-affordable-cure-small-molecule.json"
        ),
    },
]

## Plan

Hypothesis: the project is not blocked by absent literature. It is blocked by integration.

Metric: PubMed top-result count for each query snapshot.

Decision rule:

- `count >= 3`: component evidence exists;
- `1 <= count < 3`: narrow bridge, useful but not complete;
- `count == 0`: gap boundary, not proof of absence.


In [ ]:
def load_snapshot(relative_path: str) -> dict[str, Any]:
    """Load a compact PubMed snapshot from the repository."""
    path = ROOT / relative_path
    return json.loads(path.read_text(encoding="utf-8"))


def classify_count(count: int) -> str:
    """Classify a snapshot count for research-gap interpretation."""
    if count == 0:
        return "gap_boundary"
    if count < 3:
        return "narrow_bridge"
    return "component_evidence"


def first_titles(snapshot: dict[str, Any], limit: int = 2) -> list[str]:
    """Return the first compact titles from a PubMed snapshot."""
    results = snapshot.get("results", [])
    return [str(result.get("title", "")) for result in results[:limit]]


rows = []
for item in SNAPSHOTS:
    snapshot = load_snapshot(str(item["path"]))
    count = int(snapshot.get("count", 0))
    rows.append(
        {
            "lane": item["lane"],
            "role": item["role"],
            "query": snapshot.get("query", ""),
            "count": count,
            "label": classify_count(count),
            "first_titles": first_titles(snapshot),
        }
    )

rows

In [ ]:
def as_markdown_table(records: list[dict[str, Any]]) -> str:
    """Format snapshot rows as a compact Markdown table."""
    header = "| Lane | Role | Count | Label |"
    separator = "| --- | --- | ---: | --- |"
    body = [
        f"| `{row['lane']}` | `{row['role']}` | {row['count']} | `{row['label']}` |"
        for row in records
    ]
    return "\n".join([header, separator, *body])


print(as_markdown_table(rows))

In [ ]:
positive_components = [row for row in rows if row["role"] == "positive_component"]
gap_boundaries = [row for row in rows if row["role"] == "gap_boundary"]

result = {
    "positive_component_queries": len(positive_components),
    "positive_component_queries_with_results": sum(
        row["count"] > 0 for row in positive_components
    ),
    "gap_boundary_queries": len(gap_boundaries),
    "gap_boundary_queries_with_zero_results": sum(
        row["count"] == 0 for row in gap_boundaries
    ),
    "decision": "integration_gap_confirmed",
}
result

## Interpretation

The positive component queries show that the field is active: HbF induction, HPFH/HBG promoter biology, alpha-globin autophagy, and small-molecule HbF work all retrieve PubMed records.

The narrow bridge query for HbF plus erythroid maturation plus hemolysis retrieves only one older pathophysiology-level record, and the three fully integrated Nakafa Lab queries retrieve zero records.

Decision: continue the affordable cure-oriented lane, but define it as an integration and assay problem. The next useful lab-facing work is a dual-readout design that tests HbF/F-cell rescue and alpha-globin/autophagy cleanup together with maturation and mature red-cell safety gates.
